In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
import pooch

from careamics.config.data.ng_data_config import NGDataConfig
from careamics.config.data.normalization_config import MeanStdConfig
from careamics.config.data.patching_strategies import RandomPatchingConfig
from careamics.dataset_ng.microsplit_dataset import MicroSplitDataset
from careamics.dataset_ng.patching_strategies import FixedPatchingStrategy
from careamics.lvae_training.dataset import LCMultiChDloader, MicroSplitDataConfig
from careamics.lvae_training.dataset.types import DataSplitType, DataType, TilingMode

In [ ]:
DATA = pooch.create(
    path="./data/",
    base_url="https://download.fht.org/jug/msplit/ht_lif24/data_tiff/",
    registry={"ht_lif24_5ms_reduced.zip": None},
)
for fname in DATA.registry:
    DATA.fetch(fname, processor=pooch.Unzip(), progressbar=True)

DATA_PATH = DATA.abspath / (DATA.registry_files[0] + ".unzip/5ms/data/")

In [ ]:
ch0_files = sorted((DATA_PATH / "microtubules").glob("*.tiff"))
ch1_files = sorted((DATA_PATH / "nucleus").glob("*.tiff"))
rng = np.random.default_rng(42)
ch0_path = ch0_files[int(rng.integers(len(ch0_files)))]
ch1_path = ch1_files[int(rng.integers(len(ch1_files)))]
stack_ch0 = tifffile.imread(ch0_path).astype(np.float32)
stack_ch1 = tifffile.imread(ch1_path).astype(np.float32)
slice_idx = int(rng.integers(stack_ch0.shape[0]))
img_ch0 = stack_ch0[slice_idx]
img_ch1 = stack_ch1[slice_idx]
print(f"ch0={ch0_path.name} ch1={ch1_path.name} slice={slice_idx} shape={img_ch0.shape}")

In [ ]:
img_ch0 = img_ch0[np.newaxis]
img_ch1 = img_ch1[np.newaxis]

nhwc = np.stack([img_ch0, img_ch1], axis=-1)
scyx = nhwc.transpose(0, 3, 1, 2).copy()

PATCH = 64
MULTISCALE = 3

In [ ]:
def load_data_fn(data_config, datapath, datasplit_type, **kwargs):
    return nhwc

patch_seed = 17
np.random.seed(patch_seed)
coords = [int(np.random.choice(img_ch0.shape[-2] - PATCH)), int(np.random.choice(img_ch0.shape[-1] - PATCH))]

old_config = MicroSplitDataConfig(
    data_type=DataType.HTLIF24Data,
    axes="SYX",
    image_size=(PATCH, PATCH),
    grid_size=PATCH,
    num_channels=2,
    batch_size=1,
    datasplit_type=DataSplitType.Train,
    multiscale_lowres_count=MULTISCALE,
    tiling_mode=TilingMode.ShiftBoundary,
    enable_random_cropping=True,
    target_separate_normalization=True,
    train_dataloader_params={"num_workers": 0, "shuffle": True},
    val_dataloader_params={"num_workers": 0},
)

old_dset = LCMultiChDloader(
    data_config=old_config,
    datapath=Path("/synthetic"),
    load_data_fn=load_data_fn,
)
mean_d, std_d = old_dset.compute_mean_std()
old_dset.set_mean_std(mean_d, std_d)
np.random.seed(patch_seed)
old_inp, old_tgt = old_dset[0]
coords

In [ ]:
fixed_spec = {
    "data_idx": 0,
    "sample_idx": 0,
    "coords": coords,
    "patch_size": [PATCH, PATCH],
}

new_config = NGDataConfig(
    mode="training",
    data_type="array",
    axes="SCYX",
    patching=RandomPatchingConfig(patch_size=[PATCH, PATCH], seed=1),
    normalization=MeanStdConfig(per_channel=True),
    batch_size=1,
    augmentations=[],
    train_dataloader_params={"shuffle": True, "num_workers": 0},
)

new_dset = MicroSplitDataset(
    data_config=new_config,
    train_data=scyx,
    multiscale_count=MULTISCALE,
    padding_mode="reflect",
    input_is_sum=False,
    mix_uncorrelated_channels=False,
    patching_strategy=FixedPatchingStrategy([fixed_spec]),
    seed=1,
)

new_inp_region, new_tgt_region = new_dset[0]
new_inp = new_inp_region.data
new_tgt = new_tgt_region.data

In [ ]:
old_input_mean = float(mean_d["input"].mean())
old_input_std = float(std_d["input"].mean())
old_target_mean = mean_d["target"].squeeze().astype(float)
old_target_std = std_d["target"].squeeze().astype(float)
new_input_mean = np.array(new_dset._norm_input.input_means, dtype=float)
new_input_std = np.array(new_dset._norm_input.input_stds, dtype=float)
new_target_mean = np.array(new_dset._norm_target.input_means, dtype=float)
new_target_std = np.array(new_dset._norm_target.input_stds, dtype=float)

print("coords", coords)
print("input mean", old_input_mean, new_input_mean)
print("input std", old_input_std, new_input_std)
print("target mean", old_target_mean, new_target_mean)
print("target std", old_target_std, new_target_std)
print("stats close", np.allclose(old_input_mean, new_input_mean), np.allclose(old_input_std, new_input_std), np.allclose(old_target_mean, new_target_mean), np.allclose(old_target_std, new_target_std))
print("max abs input diff", [float(np.max(np.abs(old_inp[s] - new_inp[s]))) for s in range(MULTISCALE)])
print("max abs target diff", [float(np.max(np.abs(old_tgt[c] - new_tgt[c]))) for c in range(2)])

In [ ]:
fig, axes = plt.subplots(2, MULTISCALE + 2, figsize=(4 * (MULTISCALE + 2), 8))

for scale in range(MULTISCALE):
    axes[0, scale].imshow(old_inp[scale], cmap="gray")
    axes[0, scale].set_title(f"Old  inp  scale {scale}")
    axes[0, scale].axis("off")

    axes[1, scale].imshow(new_inp[scale], cmap="gray")
    axes[1, scale].set_title(f"New  inp  scale {scale}")
    axes[1, scale].axis("off")

for ch in range(2):
    axes[0, MULTISCALE + ch].imshow(old_tgt[ch], cmap="gray")
    axes[0, MULTISCALE + ch].set_title(f"Old  tgt  ch{ch}")
    axes[0, MULTISCALE + ch].axis("off")

    axes[1, MULTISCALE + ch].imshow(new_tgt[ch], cmap="gray")
    axes[1, MULTISCALE + ch].set_title(f"New  tgt  ch{ch}")
    axes[1, MULTISCALE + ch].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
_, ax = plt.subplots(1, 3)

for scale in range(MULTISCALE):
    ax[scale].imshow(old_inp[scale] - new_inp[scale])